# P-Values
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** the p-value precisely: the probability of obtaining results this extreme or more extreme, assuming H₀ is true
- **Explain** why a small p-value is evidence against H₀ but is not the probability that H₀ is false
- **Interpret** one-tailed versus two-tailed p-values and when each is appropriate

> **Tip:** Start by moving the **test statistic slider** from 0 outward, watch the shaded tail area shrink as the statistic moves further from the center. That shrinking area is the p-value.

---
## How we got here

In *17: Hypothesis Testing* we introduced the decision framework: compute a test statistic, compare it to a critical value, and either reject or fail to reject H₀. The p-value is the finer-grained quantity that drives that decision, rather than just checking whether the statistic crossed a threshold, the p-value tells us exactly how surprising the observed data is under H₀.

---
## Why this matters for data science

P-values are the most widely reported (and most widely misunderstood) statistic in all of science and data science. Every A/B test report, every academic paper, and every medical trial result includes p-values. Knowing what they actually measure, and critically, what they do not, will make you a clearer thinker and a more skeptical consumer of results throughout your career.

---
## Try it yourself

In [ ]:
from tkh_utils import make_slider, make_toggle, make_output, display_widget, register_observer

_x_range = np.linspace(-4.5, 4.5, 500)

out = make_output()
z_slider    = make_slider(-4.0, 4.0, 0.05, 1.96, "Test statistic (z):")
tail_toggle = make_toggle(["One-tailed", "Two-tailed"], "Two-tailed", "Test type:")

def render(z_val, tail_type):
    y_curve = stats.norm.pdf(_x_range)

    if tail_type == "One-tailed":
        p = 1 - stats.norm.cdf(z_val)
        x_shade = _x_range[_x_range >= z_val]
        shade_segs = [(x_shade, stats.norm.pdf(x_shade))]
    else:
        z_abs = abs(z_val)
        p = 2 * (1 - stats.norm.cdf(z_abs))
        x_right = _x_range[_x_range >=  z_abs]
        x_left  = _x_range[_x_range <= -z_abs]
        shade_segs = [
            (x_right, stats.norm.pdf(x_right)),
            (x_left,  stats.norm.pdf(x_left)),
        ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=_x_range, y=y_curve,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name="Standard Normal",
    ))
    for i, (xs, ys) in enumerate(shade_segs):
        if len(xs) > 1:
            fig.add_trace(go.Scatter(
                x=np.concatenate([[xs[0]], xs, [xs[-1]]]),
                y=np.concatenate([[0], ys, [0]]),
                fill="toself",
                fillcolor="rgba(247,103,7,0.35)",
                line=dict(width=0),
                name=f"p-value = {p:.4f}" if i == 0 else "",
                showlegend=(i == 0),
            ))
    fig.add_vline(
        x=z_val,
        line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
        annotation_text=f"z = {z_val:.2f}",
    )
    layout = base_layout(
        title=f"p-value = {p:.4f}  ({tail_type})",
        xaxis_title="Test Statistic (z)",
        yaxis_title="Probability Density under H₀",
    )
    layout.update(xaxis=dict(range=[-4.5, 4.5]))
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer([z_slider, tail_toggle], lambda change: render(z_slider.value, tail_toggle.value))
display_widget([z_slider, tail_toggle], out)
render(z_slider.value, tail_toggle.value)


---
## What's happening?

The p-value is the probability of obtaining a test statistic at least as extreme as the one observed, **assuming the null hypothesis is true**.

| Concept | Meaning |
|---------|---------|
| p-value | P(data this extreme or more \| H₀ is true) |
| One-tailed p | Area in one tail: used when Hₐ specifies a direction |
| Two-tailed p | Area in both tails: used when Hₐ is just "different from H₀" |
| p < α | Reject H₀: data is too extreme to explain by chance at level α |
| p ≥ α | Fail to reject H₀: data is consistent with H₀ (not proof H₀ is true) |

$$p_{\text{two-tailed}} = 2 \cdot P(Z \ge |z_{\text{obs}}|)$$

### What p-values do not mean

A p-value of 0.03 does **not** mean there is a 3% chance H₀ is true. It does not measure the probability the result is real, the size of the effect, or the importance of the finding. It only measures one thing: how surprising is the observed test statistic, given H₀ is true?

Return to the widget and set z to +2 for both one-tailed and two-tailed modes, confirm that the two-tailed p-value is exactly double the one-tailed value.

---
## Real-world example: Clinical trial for a new drug

A pharmaceutical company tests whether a new blood pressure medication lowers systolic BP more than a placebo. They measure the difference in means across 200 patients and compute a test statistic of z = 2.31.

The chart visualizes the p-value for this result, the probability of seeing a difference this large (or larger) if the drug actually had no effect. Notice:

- **Notice:** The shaded area (p-value) is small but not tiny, a rare but not impossible result under H₀
- **Notice:** The two-tailed shading includes both extremes, the drug lowering OR raising BP by this much would be equally "surprising" under H₀
- **Notice:** p < 0.05 in this case, which typically triggers the decision to reject H₀ and conclude the drug has a real effect

> **Discussion question:** The study produces p = 0.049, just barely under 0.05. A critic argues the result is "borderline" and should not be trusted. A defender says "the threshold is 0.05 and we cleared it." Who is right, and what does a p-value of 0.049 versus 0.051 actually tell you about the strength of evidence?

In [3]:
# ── Clinical trial p-value visualization ─────────────────────────────────────
z_obs = 2.31   # observed test statistic from a BP reduction study

x_z = np.linspace(-4, 4, 500)
y_z = stats.norm.pdf(x_z)
p_two = 2 * (1 - stats.norm.cdf(abs(z_obs)))

x_right = x_z[x_z >=  abs(z_obs)]
x_left  = x_z[x_z <= -abs(z_obs)]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_z, y=y_z, mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5),
    name="Null distribution",
))
for x_shade, label in [(x_right, f"p/2 = {p_two/2:.4f}"), (x_left, f"p/2 = {p_two/2:.4f}")]:
    fig.add_trace(go.Scatter(
        x=np.concatenate([[x_shade[0]], x_shade, [x_shade[-1]]]),
        y=np.concatenate([[0], stats.norm.pdf(x_shade), [0]]),
        fill="toself", fillcolor="rgba(247,103,7,0.30)",
        line=dict(color=PALETTE["secondary"], width=0),
        name=label, showlegend=(x_shade is x_right),
    ))
fig.add_vline(x= z_obs, line_color=PALETTE["secondary"], line_dash="dash",
              annotation_text=f"z = +{z_obs}")
fig.add_vline(x=-z_obs, line_color=PALETTE["secondary"], line_dash="dash",
              annotation_text=f"z = −{z_obs}")
layout = base_layout(
    title=f"Two-Tailed P-Value = {p_two:.4f}  (z = {z_obs}, α = 0.05)",
    xaxis_title="Test Statistic (z)",
    yaxis_title="Probability Density under H₀",
)
layout.update(xaxis=dict(range=[-4, 4]))
fig.update_layout(layout)
fig.show()

### Common p-value misconceptions: cleared up

| Misconception | Reality |
|--------------|---------|
| p = 0.03 means H₀ is 3% likely to be true | p-values say nothing about probability H₀ is true |
| p > 0.05 proves H₀ | Failing to reject ≠ proving H₀ is correct |
| Smaller p = larger effect | A tiny effect in a huge sample can have p < 0.0001 |
| p = 0.049 is very different from p = 0.051 | These represent nearly identical evidence strengths |
| p-values are objective | The choice of α, the test, and the hypotheses are all human decisions |

---
## Caution: the multiple testing problem

Every time you run a hypothesis test at α = 0.05, you accept a 5% chance of a false positive — rejecting H₀ when it is actually true. That is acceptable for a single test. But data science often involves running **many tests at once**: testing 50 features for significance, sweeping a hyperparameter grid, or running weekly A/B test checks.

If you run 20 independent tests each at α = 0.05, the probability of at least one false positive is:

$$P(\text{at least one false positive}) = 1 - (1 - 0.05)^{20} \approx 64\%$$

You are more likely than not to find a "significant" result that is pure noise.

### The Bonferroni correction

The simplest fix is to divide your α threshold by the number of tests:

$$\alpha_{\text{adjusted}} = \frac{\alpha}{m}$$

So if you run 20 tests and want an overall false-positive rate of 5%, each individual test should use α = 0.05 / 20 = 0.0025. Bonferroni is conservative but easy to apply and widely trusted.

| Situation | Approach |
|-----------|----------|
| Testing all features for correlation with target | Apply Bonferroni or FDR correction |
| Checking A/B test results daily (peeking) | Use sequential testing methods |
| Grid search over hyperparameters | Effect size matters more than p-values here |
| Running the same test across many subgroups | Bonferroni or Benjamini-Hochberg |

> **Bottom line:** whenever you see a p-value, ask how many tests were run to find it. A single p = 0.03 after testing one hypothesis is meaningful. A p = 0.03 after testing 100 features is almost certainly noise.

---
## Key takeaway

> **A p-value measures how surprising your data is under the null hypothesis — not the probability the null is true, not the size of the effect, and not whether the result matters in practice.**

---
*Next up: Type I & Type II Errors — what happens when the decision to reject (or not) is wrong, and how to balance the two kinds of mistakes*